<a href="https://colab.research.google.com/github/ihstepura/IB9AU_/blob/main/Task12_Floorplan_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Floorplan GAN — Generating Synthetic Floorplans with DCGAN

This notebook trains a **Deep Convolutional GAN (DCGAN)** on a dataset of 1000 floorplan images to generate novel, synthetic floorplan layouts.

## Why DCGAN instead of a simple MLP GAN?

The original `SD3_GAN_Illustration.ipynb` uses an MLP-based GAN for 28×28 grayscale MNIST digits (784 pixels). Our floorplan images are **128×128 RGB** — flattening them would create 49,152-dimensional vectors, making an MLP GAN impractical. DCGAN uses **strided convolutions** that naturally capture spatial structure (walls, rooms, corridors), producing much higher-quality results.

### Key Differences from the MNIST GAN:

| Feature | MNIST GAN | Floorplan DCGAN |
| :--- | :--- | :--- |
| **Architecture** | Fully connected (MLP) | Convolutional (DCGAN) |
| **Input images** | 28×28 grayscale | 128×128 RGB → resized to 64×64 |
| **Generator** | Linear layers | Transposed convolutions (upsampling) |
| **Discriminator** | Linear layers | Strided convolutions (downsampling) |
| **Latent dim** | 100 | 128 |
| **Batch normalization** | No | Yes (critical for DCGAN stability) |

In [ ]:
%%capture
!pip install torch torchvision matplotlib Pillow

## 1. Imports & Hyperparameters

- **`image_size = 64`**: We resize from 128 to 64 — standard for DCGAN and enables faster training.
- **`latent_dim = 128`**: Size of the random noise vector fed to the Generator.
- **`lr = 0.0002, beta1 = 0.5`**: Adam optimizer settings recommended by the DCGAN paper.
- **`nc = 3`**: Number of color channels (RGB).
- **`ngf / ndf = 64`**: Base filter count for Generator / Discriminator feature maps.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
import shutil

# Hyperparameters
image_size = 64       # Resize images to 64x64
latent_dim = 128      # Size of latent noise vector (z)
nc = 3                # Number of color channels (RGB)
ngf = 64              # Generator feature map base size
ndf = 64              # Discriminator feature map base size
lr = 0.0002           # Learning rate (DCGAN paper recommendation)
beta1 = 0.5           # Adam beta1 (DCGAN paper recommendation)
batch_size = 32       # Batch size
epochs = 100          # Number of training epochs

# Device selection
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

## 2. Data Loading

We use `torchvision.datasets.ImageFolder`, which expects images inside a class subfolder. Since floorplans are unconditional (no classes), we create a wrapper directory structure:
```
floorplan_dataset/
  └── floorplans/  →  (symlink to actual images)
```

Transforms:
- **Resize(64)** — Downsample from 128×128 to 64×64
- **CenterCrop(64)** — Ensure exact 64×64 dimensions
- **ToTensor()** — Convert to [0, 1] tensor
- **Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))** — Scale to [-1, 1] to match Generator's `Tanh` output

In [ ]:
# Path to the floorplan images
floorplan_image_dir = 'floorplans_v2-20251223T170650Z-3-001/floorplans_v2'

# ImageFolder requires images to be in a class subfolder.
# Create a wrapper directory with a symlink to the actual images folder.
wrapper_dir = 'floorplan_dataset'
os.makedirs(wrapper_dir, exist_ok=True)

symlink_target = os.path.join(wrapper_dir, 'floorplans')
if not os.path.exists(symlink_target):
    # Use absolute path for symlink target
    abs_image_dir = os.path.abspath(floorplan_image_dir)
    os.symlink(abs_image_dir, symlink_target)
    print(f"Created symlink: {symlink_target} -> {abs_image_dir}")
else:
    print(f"Symlink already exists: {symlink_target}")

# Data transforms
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # Scale to [-1, 1]
])

# Load dataset
dataset = datasets.ImageFolder(root=wrapper_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)

print(f"Loaded {len(dataset)} floorplan images")
print(f"Batches per epoch: {len(dataloader)}")

# Visualize a few training samples
real_batch = next(iter(dataloader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle('Sample Training Floorplans', fontsize=14)
for i, ax in enumerate(axes.flat):
    # Denormalize from [-1,1] to [0,1]
    img = real_batch[0][i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. DCGAN Architecture

### Weight Initialization
Following the DCGAN paper, all convolutional and batch normalization layers are initialized from a normal distribution:
- **Conv / ConvTranspose weights**: `N(0, 0.02)`
- **BatchNorm weights**: `N(1.0, 0.02)`, biases set to 0

### Generator
Maps a `latent_dim`-sized noise vector to a 64×64×3 RGB image using **transposed convolutions** (upsampling):
```
z (128,) → 512×4×4 → 256×8×8 → 128×16×16 → 64×32×32 → 3×64×64
```
Each layer uses `BatchNorm2d` + `ReLU`, except the final layer which uses `Tanh`.

### Discriminator
Takes a 64×64×3 RGB image and classifies it as real or fake using **strided convolutions** (downsampling):
```
3×64×64 → 64×32×32 → 128×16×16 → 256×8×8 → 512×4×4 → 1×1×1
```
Each layer uses `BatchNorm2d` + `LeakyReLU(0.2)`, except the first layer (no batchnorm) and the final layer (`Sigmoid`).

In [ ]:
# Weight initialization (DCGAN paper)
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    """DCGAN Generator: maps latent vector z to 64x64 RGB image."""
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # Input: z (latent_dim x 1 x 1)
            # Layer 1: latent_dim -> ngf*8 (512) | 4x4
            nn.ConvTranspose2d(latent_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),

            # Layer 2: ngf*8 -> ngf*4 (256) | 8x8
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            # Layer 3: ngf*4 -> ngf*2 (128) | 16x16
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            # Layer 4: ngf*2 -> ngf (64) | 32x32
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            # Layer 5: ngf -> nc (3) | 64x64
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()  # Output in [-1, 1]
        )

    def forward(self, z):
        return self.main(z)


class Discriminator(nn.Module):
    """DCGAN Discriminator: classifies 64x64 RGB images as real/fake."""
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            # Input: nc x 64 x 64
            # Layer 1: nc -> ndf (64) | 32x32 (no batchnorm per DCGAN paper)
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 2: ndf -> ndf*2 (128) | 16x16
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 3: ndf*2 -> ndf*4 (256) | 8x8
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 4: ndf*4 -> ndf*8 (512) | 4x4
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 5: ndf*8 -> 1 | 1x1
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.main(img).view(-1, 1)


# Initialize networks
G = Generator().to(device)
D = Discriminator().to(device)

# Apply weight initialization
G.apply(weights_init)
D.apply(weights_init)

# Print architectures
print("=" * 60)
print("GENERATOR")
print("=" * 60)
print(G)
print(f"\nTotal parameters: {sum(p.numel() for p in G.parameters()):,}")

print("\n" + "=" * 60)
print("DISCRIMINATOR")
print("=" * 60)
print(D)
print(f"\nTotal parameters: {sum(p.numel() for p in D.parameters()):,}")

## 4. Training Loop

The training follows the standard GAN minimax game:

**For each batch:**

1. **Train Discriminator (D)**:
   - Forward pass real images → D should output ~1 (real)
   - Generate fake images with G, forward through D → D should output ~0 (fake)
   - Compute total D loss = `loss_real + loss_fake`
   - Update D weights only

2. **Train Generator (G)**:
   - Generate new fake images, forward through D
   - G wants D to output ~1 (fool the discriminator)
   - Compute G loss and update G weights only

We also keep a **fixed noise vector** to track visual progress across epochs.

In [ ]:
# Loss and Optimizers
criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(beta1, 0.999))

# Fixed noise for tracking progress
fixed_noise = torch.randn(16, latent_dim, 1, 1, device=device)

# Training history
G_losses = []
D_losses = []

# Label convention
real_label = 1.0
fake_label = 0.0

print(f"Starting training for {epochs} epochs...")
print(f"Dataset: {len(dataset)} images | Batch size: {batch_size} | Batches/epoch: {len(dataloader)}")
print("-" * 70)

for epoch in range(epochs):
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        b_size = real_imgs.size(0)

        # ============================================
        # (1) Train Discriminator: maximize log(D(x)) + log(1 - D(G(z)))
        # ============================================
        optimizer_D.zero_grad()

        # Train with real images
        labels_real = torch.full((b_size, 1), real_label, device=device)
        output_real = D(real_imgs)
        loss_D_real = criterion(output_real, labels_real)
        loss_D_real.backward()

        # Train with fake images
        z = torch.randn(b_size, latent_dim, 1, 1, device=device)
        fake_imgs = G(z)
        labels_fake = torch.full((b_size, 1), fake_label, device=device)
        output_fake = D(fake_imgs.detach())  # Detach to avoid training G
        loss_D_fake = criterion(output_fake, labels_fake)
        loss_D_fake.backward()

        loss_D = loss_D_real + loss_D_fake
        optimizer_D.step()

        # ============================================
        # (2) Train Generator: maximize log(D(G(z)))
        # ============================================
        optimizer_G.zero_grad()

        z = torch.randn(b_size, latent_dim, 1, 1, device=device)
        fake_imgs = G(z)
        output = D(fake_imgs)
        # Generator wants discriminator to think fakes are real
        loss_G = criterion(output, labels_real)
        loss_G.backward()
        optimizer_G.step()

        # Record losses
        G_losses.append(loss_G.item())
        D_losses.append(loss_D.item())

    # Print epoch summary
    print(f"Epoch [{epoch+1:3d}/{epochs}] | Loss D: {loss_D.item():.4f} | Loss G: {loss_G.item():.4f}")

    # Show progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            fake_progress = G(fixed_noise).detach().cpu()
        fig, axes = plt.subplots(2, 4, figsize=(12, 6))
        fig.suptitle(f'Generated Floorplans — Epoch {epoch+1}', fontsize=14)
        for j, ax in enumerate(axes.flat):
            img = fake_progress[j].permute(1, 2, 0).numpy() * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            ax.imshow(img)
            ax.axis('off')
        plt.tight_layout()
        plt.show()

print("\n" + "=" * 70)
print("Training complete!")

## 5. Training Loss Visualization

Plotting the Generator and Discriminator losses over training helps diagnose training health:
- **Healthy training**: D loss oscillates near 0.5–1.0, G loss gradually decreases
- **Mode collapse**: G loss drops very low while D loss spikes (G found a single pattern to fool D)
- **D dominance**: D loss → 0 while G loss → high (D is too strong, G can't learn)

In [ ]:
# Plot training losses
plt.figure(figsize=(12, 5))
plt.plot(G_losses, label='Generator Loss', alpha=0.7, color='#2196F3')
plt.plot(D_losses, label='Discriminator Loss', alpha=0.7, color='#FF5722')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.title('Generator and Discriminator Loss During Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('floorplan_gan_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print("Loss plot saved to 'floorplan_gan_loss.png'")

## 6. Generate Synthetic Floorplans

Now that the Generator is trained, we sample random noise vectors and generate a grid of **16 synthetic floorplan images**. Each image is unique — generated entirely from random noise, not copied from the training data.

In [ ]:
# Generate a 4x4 grid of synthetic floorplans
G.eval()
with torch.no_grad():
    z = torch.randn(16, latent_dim, 1, 1, device=device)
    generated = G(z).detach().cpu()

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Generated Synthetic Floorplans', fontsize=16, fontweight='bold')
for i, ax in enumerate(axes.flat):
    # Denormalize from [-1,1] to [0,1]
    img = generated[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f'Sample {i+1}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig('generated_floorplans.png', dpi=150, bbox_inches='tight')
plt.show()
print("Generated floorplans saved to 'generated_floorplans.png'")

## 7. Real vs. Generated Comparison

Side-by-side comparison of real training images and generated samples to visually assess the Generator's quality.

In [ ]:
# Side-by-side comparison: Real vs Generated
fig, axes = plt.subplots(2, 8, figsize=(18, 5))

# Top row: Real images
real_batch = next(iter(dataloader))
for i in range(8):
    img = real_batch[0][i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[0, i].imshow(img)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('REAL', fontsize=12, fontweight='bold', color='green')

# Bottom row: Generated images
G.eval()
with torch.no_grad():
    z = torch.randn(8, latent_dim, 1, 1, device=device)
    fake_imgs = G(z).detach().cpu()
for i in range(8):
    img = fake_imgs[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    axes[1, i].imshow(img)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('GENERATED', fontsize=12, fontweight='bold', color='blue')

plt.suptitle('Real vs. Generated Floorplans', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('real_vs_generated.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Trained Model

Save the Generator weights so we can load them later to generate new floorplans without retraining.

In [ ]:
# Save the trained Generator
save_path = 'floorplan_generator.pth'
torch.save({
    'generator_state_dict': G.state_dict(),
    'discriminator_state_dict': D.state_dict(),
    'optimizer_G_state_dict': optimizer_G.state_dict(),
    'optimizer_D_state_dict': optimizer_D.state_dict(),
    'epoch': epochs,
    'G_losses': G_losses,
    'D_losses': D_losses,
    'latent_dim': latent_dim,
    'image_size': image_size,
}, save_path)

print(f"Model saved to '{save_path}'")
print(f"File size: {os.path.getsize(save_path) / (1024*1024):.1f} MB")

## 9. Load & Generate (Standalone)

This cell can be run independently to load a previously trained generator and create new floorplans. Just make sure the `floorplan_generator.pth` file exists in the current directory.

In [ ]:
# ===== Standalone: Load trained Generator and generate floorplans =====

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Load checkpoint
checkpoint = torch.load('floorplan_generator.pth', map_location='cpu', weights_only=False)
latent_dim_loaded = checkpoint['latent_dim']
ngf_loaded = 64
nc_loaded = 3

# Rebuild Generator
class GeneratorLoaded(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim_loaded, ngf_loaded * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf_loaded * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf_loaded * 8, ngf_loaded * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf_loaded * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf_loaded * 4, ngf_loaded * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf_loaded * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf_loaded * 2, ngf_loaded, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf_loaded),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf_loaded, nc_loaded, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, z):
        return self.main(z)

G_loaded = GeneratorLoaded()
G_loaded.load_state_dict(checkpoint['generator_state_dict'])
G_loaded.eval()

# Generate new floorplans
num_samples = 16
with torch.no_grad():
    z = torch.randn(num_samples, latent_dim_loaded, 1, 1)
    generated = G_loaded(z)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Newly Generated Floorplans (from saved model)', fontsize=16, fontweight='bold')
for i, ax in enumerate(axes.flat):
    img = generated[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f'Floorplan {i+1}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Generated {num_samples} new floorplans from saved model.")